# Media Intensity Analysis - 3 Conflicts

**Course:** 42578 Advanced Business Analytics (F26) - DTU
**Project:** War Media Impact on U.S. Financial Markets
**Dataset:** `all_conflicts_combined_2.csv`

---

This notebook produces:

1. **Clean load** with per-conflict keyword handling
2. **Per-conflict bell curves** - one panel per conflict
3. **Keyword decomposition** per conflict (auto-detects which keywords have data)
4. **Normalised overlay** - all three conflicts on one chart, aligned by peak day
5. **Absolute overlay** for magnitude comparison
6. **Decay metrics** - peak date, rise time, half-life, 20%-fade

**Important note on the metric.** `war_pct` counts article-keyword mentions, not unique articles. An article mentioning `israel`, `hamas`, and `gaza` is counted three times, not once. This means `war_pct` can exceed 100% and should be read as a **relative media intensity index**, not a literal share of news. Curve shapes (rise rate, peak day, decay rate) are unaffected because the double-count factor is approximately constant across days.


## 1. Setup

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

CSV_PATH = "all_conflicts_combined_2.csv"
OUT_DIR = Path("figures")
OUT_DIR.mkdir(exist_ok=True)

CONFLICTS = ["Russia_Ukraine", "Israel_Gaza", "US_Israel_Iran"]

COLOUR = {
    "Russia_Ukraine":   "#1f77b4",
    "Israel_Gaza":      "#d62728",
    "US_Israel_Iran":   "#2ca02c",
}
LABEL = {
    "Russia_Ukraine":   "Russia-Ukraine (2022)",
    "Israel_Gaza":      "Israel-Gaza (2023-24)",
    "US_Israel_Iran":   "US-Israel-Iran (2026)",
}

# The metric's true name, used on all y-axis labels
INTENSITY_LABEL = "Media intensity index (keyword mentions per 100 articles)"
INTENSITY_SHORT = "Intensity index"  # for overlay charts

## 2. Load and clean

NaN in a keyword column means "that term wasn't tracked for this conflict" - we never fill those with zero because it would falsely imply we measured no coverage when we didn't measure at all.

In [3]:
df = pd.read_csv(CSV_PATH, parse_dates=["date"])
df = df.sort_values(["conflict", "date"]).reset_index(drop=True)

print(f"Loaded {len(df)} rows across {df['conflict'].nunique()} conflicts")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}\n")

for c in CONFLICTS:
    sub = df[df["conflict"] == c]
    if len(sub) == 0:
        print(f"  {c:22s} NOT in dataset")
        continue
    print(f"  {c:22s} {len(sub):4d} days   "
          f"{sub['date'].min().date()} -> {sub['date'].max().date()}")

FileNotFoundError: [Errno 2] No such file or directory: 'all_conflicts_combined_2.csv'

In [ ]:
bad = df[df["total_articles"] <= 0]
print(f"Rows with total_articles <= 0: {len(bad)}")

check = ((df["war_articles"] / df["total_articles"] * 100) - df["war_pct"]).abs()
print(f"war_pct vs war_articles/total_articles (max dev): {check.max():.4f}")

print(f"Rows with missing war_pct_7d: {df['war_pct_7d'].isna().sum()}")

## 3. Demonstrate the double-count issue

The `war_pct` column in the raw data exceeds 100% on high-intensity days for every conflict. This happens because `war_articles` counts keyword matches, not unique articles — an article mentioning `israel`, `hamas`, and `gaza` is counted 3 times.

The cell below shows the max daily `war_pct` per conflict and compares it to the share of total articles, making the inflation visible.

In [ ]:
kw_cols = [c for c in df.columns if c.startswith("kw_")]

print("Inflation diagnostic - maximum daily war_pct per conflict:\n")
print(f"{'Conflict':<25} {'max war_pct':>12} {'total articles':>16}  {'date':>12}")
print("-" * 72)

for conflict in CONFLICTS:
    sub = df[df["conflict"] == conflict]
    max_idx = sub["war_pct"].idxmax()
    peak_row = sub.loc[max_idx]
    print(f"{LABEL[conflict]:<25} {peak_row['war_pct']:>12.2f} "
          f"{peak_row['total_articles']:>16,.0f}  "
          f"{peak_row['date'].date()}")

print("\nValues above 100% confirm that war_articles counts keyword matches,")
print("not unique articles. An article mentioning 'israel', 'hamas', and 'gaza'")
print("is counted 3 times. This means war_pct is best read as a RELATIVE")
print("INTENSITY INDEX, not a literal percentage of news.")
print()
print("Curve SHAPES (rise/decay rates) are unaffected because the double-count")
print("factor is approximately constant day-to-day.")

## 4. Auto-detect keyword columns per conflict

We scan each conflict's rows and keep any `kw_*` column with actual data. When you re-run the upstream scraper with more terms, this cell picks them up automatically as long as columns follow the `kw_<term>` convention.

In [ ]:
KEYWORDS = {}
for conflict in CONFLICTS:
    sub = df[df["conflict"] == conflict]
    if len(sub) == 0:
        KEYWORDS[conflict] = []
        continue
    active = [kw for kw in kw_cols
              if kw in sub.columns
              and sub[kw].notna().any()
              and sub[kw].sum() > 0]
    KEYWORDS[conflict] = active

for conflict in CONFLICTS:
    n = len(KEYWORDS[conflict])
    terms = [k.replace("kw_", "") for k in KEYWORDS[conflict]]
    print(f"  {conflict:22s} {n:2d} keywords: {', '.join(terms)}")

## 5. Per-conflict intensity bell curves

Each conflict gets its own panel: daily intensity with a 7-day rolling mean overlaid, peak day annotated.

In [ ]:
def plot_conflict_intensity(sub, conflict_key, save=True):
    fig, ax = plt.subplots()
    dates = sub["date"]
    ax.fill_between(dates, 0, sub["war_pct"], alpha=0.25,
                    color=COLOUR[conflict_key], label="daily (raw)")
    ax.plot(dates, sub["war_pct_7d"], color=COLOUR[conflict_key],
            linewidth=2.5, label="7-day rolling mean")

    smooth = sub["war_pct_7d"]
    if smooth.notna().any():
        peak_idx = smooth.idxmax()
        peak_date = sub.loc[peak_idx, "date"]
        peak_val = smooth.loc[peak_idx]
        ax.axvline(peak_date, color="black", linestyle=":", alpha=0.6)
        ax.annotate(f"peak: {peak_date.date()}\nindex = {peak_val:.1f}",
                    xy=(peak_date, peak_val),
                    xytext=(12, -8), textcoords="offset points", fontsize=9,
                    bbox=dict(boxstyle="round,pad=0.4", fc="white",
                              ec="gray", alpha=0.9))

    ax.set_title(f"{LABEL[conflict_key]} - media intensity",
                 fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Date")
    ax.set_ylabel(INTENSITY_LABEL)
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d\n%Y"))
    ax.legend(loc="upper right", fontsize=10, framealpha=0.95)
    ax.set_ylim(bottom=0)
    fig.tight_layout()
    if save:
        fig.savefig(OUT_DIR / f"intensity__{conflict_key}.png",
                    dpi=140, bbox_inches="tight")
    plt.show()


for conflict in CONFLICTS:
    sub = df[df["conflict"] == conflict].reset_index(drop=True)
    if len(sub) > 0:
        plot_conflict_intensity(sub, conflict)

## 6. Keyword breakdown per conflict

For each conflict, this shows *which* specific terms drove the coverage - the narrative composition. Keywords are ordered by total mentions so the busiest terms get the most visible colours.

In [ ]:
def plot_keyword_breakdown(sub, conflict_key, save=True):
    kws = KEYWORDS[conflict_key]
    if not kws:
        print(f"  {conflict_key}: no keyword columns with data - skipping")
        return

    ordered = sorted(kws, key=lambda k: -sub[k].sum())

    fig, ax = plt.subplots(figsize=(13, 6))
    cmap = plt.cm.tab20(np.linspace(0, 1, max(len(ordered), 20)))

    for i, kw in enumerate(ordered):
        smoothed = sub[kw].rolling(7, min_periods=1).mean()
        ax.plot(sub["date"], smoothed, linewidth=1.8, color=cmap[i],
                label=kw.replace("kw_", "").replace("_", " "))

    ax.set_title(f"{LABEL[conflict_key]} - keyword decomposition "
                 f"({len(ordered)} terms, 7-day rolling)",
                 fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Date")
    ax.set_ylabel("Daily articles containing keyword")
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d\n%Y"))
    ncol = 2 if len(ordered) <= 10 else 3
    ax.legend(loc="upper right", fontsize=9, ncol=ncol, framealpha=0.95)
    ax.set_ylim(bottom=0)
    fig.tight_layout()
    if save:
        fig.savefig(OUT_DIR / f"keywords__{conflict_key}.png",
                    dpi=140, bbox_inches="tight")
    plt.show()


for conflict in CONFLICTS:
    sub = df[df["conflict"] == conflict].reset_index(drop=True)
    if len(sub) > 0:
        plot_keyword_breakdown(sub, conflict)

## 7. Normalised overlay - all conflicts on one chart

Each conflict's 7-day rolling intensity is min-max scaled to `[0, 1]` within its own window, then re-centred on its peak day. This compares **curve shapes** across conflicts regardless of magnitude or calendar date - exactly what transfer learning needs as a baseline comparison.

In [ ]:
def build_normalised_aligned(sub):
    out = sub.copy().reset_index(drop=True).dropna(subset=["war_pct_7d"])
    v = out["war_pct_7d"]
    vmin, vmax = v.min(), v.max()
    out["normalised_intensity"] = (v - vmin) / (vmax - vmin) if vmax > vmin else 0.0
    peak_date = out.loc[v.idxmax(), "date"]
    out["days_from_peak"] = (out["date"] - peak_date).dt.days
    return out


fig, ax = plt.subplots(figsize=(13, 6.5))
peak_info = {}

for conflict in CONFLICTS:
    sub = df[df["conflict"] == conflict]
    if len(sub) == 0:
        continue
    aligned = build_normalised_aligned(sub)

    ax.plot(aligned["days_from_peak"], aligned["normalised_intensity"],
            color=COLOUR[conflict], linewidth=2.2, label=LABEL[conflict])
    ax.fill_between(aligned["days_from_peak"], 0, aligned["normalised_intensity"],
                    color=COLOUR[conflict], alpha=0.12)

    peak_info[conflict] = {
        "date": aligned.loc[aligned["normalised_intensity"].idxmax(), "date"],
        "peak_val": sub["war_pct_7d"].max(),
    }

ax.axvline(0, color="black", linestyle=":", alpha=0.6, linewidth=1)
ax.text(0, 1.02, "peak", ha="center", fontsize=9)

ax.set_title("Media intensity across 3 conflicts - normalised & aligned by peak",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Days from peak coverage")
ax.set_ylabel(f"{INTENSITY_SHORT} (normalised 0-1)")
ax.legend(loc="upper right", fontsize=10, framealpha=0.95)
ax.set_ylim(-0.02, 1.05)
fig.tight_layout()
fig.savefig(OUT_DIR / "overlay__normalised_aligned.png", dpi=140, bbox_inches="tight")
plt.show()

print("\nPeak info per conflict:")
for c, info in peak_info.items():
    print(f"  {LABEL[c]:28s}  peak {info['date'].date()}  "
          f"(index = {info['peak_val']:.2f})")

## 8. Absolute-scale overlay

Same peak-day alignment, but no rescaling - so you can see which conflicts dominated more of the news cycle in absolute terms, not just shapes.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6.5))

for conflict in CONFLICTS:
    sub = df[df["conflict"] == conflict]
    if len(sub) == 0:
        continue
    aligned = build_normalised_aligned(sub)
    ax.plot(aligned["days_from_peak"], aligned["war_pct_7d"],
            color=COLOUR[conflict], linewidth=2.2, label=LABEL[conflict])
    ax.fill_between(aligned["days_from_peak"], 0, aligned["war_pct_7d"],
                    color=COLOUR[conflict], alpha=0.12)

ax.axvline(0, color="black", linestyle=":", alpha=0.6, linewidth=1)
ax.set_title("Media intensity - absolute scale",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Days from peak coverage")
ax.set_ylabel(INTENSITY_LABEL)
ax.legend(loc="upper right", fontsize=10, framealpha=0.95)
ax.set_ylim(bottom=0)
fig.tight_layout()
fig.savefig(OUT_DIR / "overlay__absolute.png", dpi=140, bbox_inches="tight")
plt.show()

## 9. Decay metrics

Peak date, rise time, half-life, and 20%-fade per conflict. If a window is too short to capture the fade, the metric comes back as `NaN`.

In [ ]:
def compute_metrics(sub, conflict_key):
    sub = sub.reset_index(drop=True)
    v = sub["war_pct_7d"]
    if v.notna().sum() == 0:
        return {"conflict": LABEL[conflict_key]}

    peak_idx = v.idxmax()
    peak_date = sub.loc[peak_idx, "date"]
    peak_val = v.iloc[peak_idx]

    rise_mask = v >= 0.10 * peak_val
    first_rise = sub.loc[rise_mask, "date"].min()
    rise_days = (peak_date - first_rise).days if pd.notna(first_rise) else None

    post = sub.loc[peak_idx:].reset_index(drop=True)
    half = post[post["war_pct_7d"] < 0.5 * peak_val]
    fifth = post[post["war_pct_7d"] < 0.2 * peak_val]

    return {
        "conflict": LABEL[conflict_key],
        "peak_date": peak_date.date(),
        "peak_intensity_index": round(peak_val, 2),
        "days_covered": len(sub),
        "rise_days_10pct_to_peak": rise_days,
        "days_to_half_peak": (half["date"].min() - peak_date).days if len(half) else None,
        "days_to_20pct_peak": (fifth["date"].min() - peak_date).days if len(fifth) else None,
    }


rows = []
for conflict in CONFLICTS:
    sub = df[df["conflict"] == conflict]
    if len(sub) > 0:
        rows.append(compute_metrics(sub, conflict))

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(OUT_DIR / "decay_metrics.csv", index=False)
metrics_df

## 10. Notes & caveats for the report

**The intensity metric is a relative index, not a percentage.** As Section 3 demonstrates, `war_articles` counts keyword-article matches, so articles mentioning multiple keywords inflate the count. Use the index for shape comparisons across conflicts (which is what the project needs). If you need a literal percentage of news, the upstream scraper must be modified to count unique articles instead of keyword matches.

**Window durations are uneven.** Russia-Ukraine 119 days, Israel-Gaza 154, US-Israel-Iran 113. When comparing decay metrics, a short window cannot report a fade that hadn't completed yet.

**Pre-invasion baseline captured for Ukraine.** The Jan-Feb 2022 pre-invasion window is long enough to see the slow buildup of Russia-Ukraine coverage before Feb 24. This is visible in the normalised overlay as the blue curve's long left tail. Useful context: even before the invasion, Ukraine-Russia tensions were generating meaningful news volume.

**October 7 was a true shock.** Israel-Gaza's pre-peak tail is near-zero, unlike Ukraine. The curve goes from noise to saturation in under 7 days. This is a genuinely different conflict type from a media-intensity standpoint (shock vs. slow-burn).

**Iran decays fastest.** Even within 113 days we can see US-Israel-Iran's half-life is ~10 days vs. ~26-29 for the other two. Worth investigating whether this reflects a genuinely shorter attention cycle or just the specific way Iran coverage was distributed.

**Keyword expansion roadmap.** Methodology targets ~20 keywords per conflict. Current counts:

- Russia-Ukraine: 10 keywords
- Israel-Gaza: 9 keywords
- US-Israel-Iran: 10 keywords

To reach 20 per conflict, re-run the upstream GDELT scraper with expanded term lists. Once the CSV has more `kw_*` columns, Section 4 detects them automatically.